In [1]:
!pip -q install mne pandas numpy scipy scikit-learn matplotlib seaborn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 86.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import os
import glob
import json
import zipfile
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("MNE:", mne.__version__)

TensorFlow: 2.20.0
MNE: 1.13.2


In [3]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to", SEED)

Random seed set to 42


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for f in uploaded:
    print(f)

In [ ]:
ZIP_PATH = "/content/EEG1920.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "EEG1920.zip was not found. Please upload the ZIP file."
    )

print("ZIP found:", ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / (1024**3), 2), "GB")

In [ ]:
EXTRACT_ROOT = "/content/bigP3BCI"

os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_ROOT)

print("Extraction completed.")
print("Dataset location:", EXTRACT_ROOT)

In [ ]:
all_files = glob.glob(
    os.path.join(EXTRACT_ROOT, "**", "*"),
    recursive=True
)

bdf_files = [
    f for f in all_files
    if f.lower().endswith(".bdf")
]

events_files = [
    f for f in all_files
    if f.lower().endswith("_events.tsv")
]

print("Total files:", len(all_files))
print("BDF files:", len(bdf_files))
print("Events files:", len(events_files))

In [ ]:
SELECTED_SUBJECTS = [
    "sub-1",
    "sub-2",
    "sub-3"
]

print("Subjects used:")
for s in SELECTED_SUBJECTS:
    print(s)

In [ ]:
recordings = []

for subject in SELECTED_SUBJECTS:

    subject_bdfs = [
        f for f in bdf_files
        if f"/{subject}/" in f
    ]

    for eeg_path in subject_bdfs:

        base = eeg_path.replace("_eeg.bdf", "")
        events_path = base + "_events.tsv"

        if os.path.exists(events_path):
            recordings.append(
                {
                    "subject": subject,
                    "eeg": eeg_path,
                    "events": events_path
                }
            )

print("Valid EEG + events recordings:", len(recordings))

for r in recordings[:10]:
    print("\nSubject:", r["subject"])
    print("EEG:", r["eeg"])
    print("Events:", r["events"])

In [ ]:
for r in recordings:

    df = pd.read_csv(
        r["events"],
        sep="\t"
    )

    print("\n" + "="*60)
    print(r["subject"])
    print(os.path.basename(r["events"]))

    print("\nColumns:")
    print(df.columns.tolist())

    if "trial_type" in df.columns:
        print("\nTrial types:")
        print(df["trial_type"].value_counts())

In [ ]:
LOW_FREQ = 1.0
HIGH_FREQ = 15.0

NOTCH_FREQ = 50.0

TARGET_SFREQ = 120

FILTER_ORDER = 6

EPOCH_TMIN = 0.0
EPOCH_SAMPLES = 120

TARGET_LABEL = 1
NONTARGET_LABEL = 0

print("Bandpass:", LOW_FREQ, "-", HIGH_FREQ, "Hz")
print("Notch:", NOTCH_FREQ, "Hz")
print("Sampling frequency:", TARGET_SFREQ, "Hz")
print("Epoch samples:", EPOCH_SAMPLES)

In [ ]:
def winsorize_signal(data, lower=0.01, upper=0.99):
    """
    Limit extreme EEG values using percentile clipping.
    """

    low = np.quantile(
        data,
        lower,
        axis=-1,
        keepdims=True
    )

    high = np.quantile(
        data,
        upper,
        axis=-1,
        keepdims=True
    )

    return np.clip(
        data,
        low,
        high
    )

In [ ]:
def butter_bandpass(data, sfreq, lowcut, highcut, order=6):

    nyquist = sfreq / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    sos = butter(
        order,
        [low, high],
        btype="bandpass",
        output="sos"
    )

    return sosfiltfilt(
        sos,
        data,
        axis=-1
    )

In [ ]:
def notch_filter(data, sfreq, notch_freq=50):

    b, a = iirnotch(
        notch_freq,
        Q=30,
        fs=sfreq
    )

    return filtfilt(
        b,
        a,
        data,
        axis=-1
    )

In [ ]:
def get_eeg_channels(raw, n_channels=16):

    eeg_picks = mne.pick_types(
        raw.info,
        eeg=True,
        exclude="bads"
    )

    names = [
        raw.ch_names[i]
        for i in eeg_picks
    ]

    if len(names) < n_channels:
        print(
            "Warning:",
            len(names),
            "EEG channels available."
        )
        return names

    return names[:n_channels]

In [ ]:
def preprocess_recording(
    eeg_path,
    events_path,
    n_channels=16
):

    print("\nProcessing:")
    print(os.path.basename(eeg_path))

    # -----------------------------------
    # 1. Load BDF
    # -----------------------------------
    raw = mne.io.read_raw_bdf(
        eeg_path,
        preload=True,
        verbose=False
    )

    print("Original sampling rate:",
          raw.info["sfreq"])

    # -----------------------------------
    # 2. Select EEG channels
    # -----------------------------------
    eeg_channels = get_eeg_channels(
        raw,
        n_channels=n_channels
    )

    raw.pick(
        eeg_channels
    )

    print("EEG channels:",
          len(raw.ch_names))

    # -----------------------------------
    # 3. Read BIDS events
    # -----------------------------------
    events_df = pd.read_csv(
        events_path,
        sep="\t"
    )

    print("Event labels:")
    print(
        events_df["trial_type"]
        .value_counts()
    )

    # -----------------------------------
    # 4. Original event samples
    # -----------------------------------
    original_sfreq = raw.info["sfreq"]

    if "sample" in events_df.columns:

        original_samples = (
            events_df["sample"]
            .astype(int)
            .values
        )

    else:

        original_samples = (
            events_df["onset"].values
            * original_sfreq
        ).astype(int)

    # -----------------------------------
    # 5. Create labels
    # -----------------------------------
    labels_text = (
        events_df["trial_type"]
        .astype(str)
        .str.lower()
        .values
    )

    event_samples = []
    event_labels = []

    for sample, label in zip(
        original_samples,
        labels_text
    ):

        if "target" in label and "nontarget" not in label:

            event_samples.append(
                int(sample)
            )

            event_labels.append(
                TARGET_LABEL
            )

        elif "nontarget" in label:

            event_samples.append(
                int(sample)
            )

            event_labels.append(
                NONTARGET_LABEL
            )

    if len(event_samples) == 0:
        print("No usable Target/NonTarget events.")
        return None, None

    event_samples = np.array(
        event_samples
    )

    event_labels = np.array(
        event_labels
    )

    # -----------------------------------
    # 6. Bandpass
    # -----------------------------------
    data = raw.get_data()

    data = butter_bandpass(
        data,
        original_sfreq,
        LOW_FREQ,
        HIGH_FREQ,
        FILTER_ORDER
    )

    # -----------------------------------
    # 7. Notch
    # -----------------------------------
    data = notch_filter(
        data,
        original_sfreq,
        NOTCH_FREQ
    )

    # -----------------------------------
    # 8. Put filtered data back
    # -----------------------------------
    raw._data = data

    # -----------------------------------
    # 9. Resample
    # -----------------------------------
    raw.resample(
        TARGET_SFREQ,
        npad="auto"
    )

    # -----------------------------------
    # 10. Convert event samples
    # -----------------------------------
    new_samples = (
        event_samples
        * TARGET_SFREQ
        / original_sfreq
    ).astype(int)

    mne_events = np.column_stack(
        [
            new_samples,
            np.zeros(len(new_samples), dtype=int),
            event_labels
        ]
    )

    # -----------------------------------
    # 11. Epoch
    # -----------------------------------
    tmax = (
        EPOCH_SAMPLES - 1
    ) / TARGET_SFREQ

    event_id = {
        "NonTarget": NONTARGET_LABEL,
        "Target": TARGET_LABEL
    }

    epochs = mne.Epochs(
        raw,
        mne_events,
        event_id=event_id,
        tmin=EPOCH_TMIN,
        tmax=tmax,
        baseline=None,
        preload=True,
        reject_by_annotation=True,
        verbose=False
    )

    X = epochs.get_data()

    # -----------------------------------
    # 12. Epoch labels
    # -----------------------------------
    y = epochs.events[:, 2]

    # -----------------------------------
    # 13. Winsorization
    # -----------------------------------
    X = winsorize_signal(X)

    print("Epoch shape:", X.shape)
    print("Target:", np.sum(y == 1))
    print("NonTarget:", np.sum(y == 0))

    return X, y

In [ ]:
TEST_RECORDING = recordings[0]

X_test_demo, y_test_demo = preprocess_recording(
    TEST_RECORDING["eeg"],
    TEST_RECORDING["events"]
)

if X_test_demo is not None:
    print("\nTest successful.")
    print("X:", X_test_demo.shape)
    print("y:", y_test_demo.shape)

In [ ]:
subject_data = {}

for subject in SELECTED_SUBJECTS:

    print("\n")
    print("#"*70)
    print("SUBJECT:", subject)
    print("#"*70)

    subject_recordings = [
        r for r in recordings
        if r["subject"] == subject
    ]

    X_all = []
    y_all = []

    for r in subject_recordings:

        X, y = preprocess_recording(
            r["eeg"],
            r["events"]
        )

        if X is not None and len(X) > 0:

            X_all.append(X)
            y_all.append(y)

    if len(X_all) > 0:

        X_subject = np.concatenate(
            X_all,
            axis=0
        )

        y_subject = np.concatenate(
            y_all,
            axis=0
        )

        subject_data[subject] = {
            "X": X_subject,
            "y": y_subject
        }

        print(
            "\nFINAL",
            subject,
            X_subject.shape,
            y_subject.shape
        )

In [ ]:
print("SUBJECT DATA SUMMARY")
print("="*60)

for subject, data in subject_data.items():

    X = data["X"]
    y = data["y"]

    print(
        subject,
        "X =", X.shape,
        "y =", y.shape,
        "Target =", np.sum(y == 1),
        "NonTarget =", np.sum(y == 0)
    )

In [ ]:
def prepare_for_model(X):

    return X[..., np.newaxis]

In [ ]:
def TCNBlock(
    x,
    filters=32,
    kernel_size=3,
    dilation_rate=1,
    dropout=0.2
):

    shortcut = x

    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate
    )(x)

    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:

        shortcut = layers.Conv1D(
            filters,
            1,
            padding="same"
        )(shortcut)

    x = layers.Add()([
        x,
        shortcut
    ])

    x = layers.Activation("elu")(x)

    return x

In [ ]:
class GaussianFuzzyBlock(layers.Layer):

    def __init__(
        self,
        units=16,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):

        features = int(input_shape[-1])

        self.centers = self.add_weight(
            name="centers",
            shape=(features, self.units),
            initializer="random_normal",
            trainable=True
        )

        self.sigmas = self.add_weight(
            name="sigmas",
            shape=(features, self.units),
            initializer="ones",
            trainable=True
        )

    def call(self, inputs):

        x = tf.expand_dims(
            inputs,
            axis=-1
        )

        centers = tf.expand_dims(
            self.centers,
            axis=0
        )

        sigmas = tf.expand_dims(
            tf.nn.softplus(self.sigmas),
            axis=0
        )

        membership = tf.exp(
            -tf.square(
                x - centers
            ) /
            (
                2.0 *
                tf.square(sigmas) +
                1e-7
            )
        )

        membership = tf.reduce_mean(
            membership,
            axis=1
        )

        return membership

In [ ]:
def build_EEG_TCFNet():

    inputs = layers.Input(
        shape=(16, 120, 1),
        name="EEG_Input"
    )

    # =========================================
    # EEG CNN / EEG-Net style feature extraction
    # =========================================

    x = layers.Conv2D(
        8,
        kernel_size=(1, 16),
        padding="same"
    )(inputs)

    x = layers.BatchNormalization()(x)

    x = layers.DepthwiseConv2D(
        kernel_size=(16, 1),
        depth_multiplier=2,
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)

    x = layers.AveragePooling2D(
        pool_size=(1, 4)
    )(x)

    x = layers.Dropout(0.25)(x)

    x = layers.SeparableConv2D(
        16,
        kernel_size=(1, 8),
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)

    x = layers.AveragePooling2D(
        pool_size=(1, 4)
    )(x)

    x = layers.Dropout(0.25)(x)

    # =========================================
    # Convert to sequence
    # =========================================

    x = layers.Permute(
        (2, 1, 3)
    )(x)

    x = layers.Reshape(
        (-1, int(x.shape[2] * x.shape[3]))
    )(x)

    # =========================================
    # TCN
    # =========================================

    x = TCNBlock(
        x,
        filters=32,
        dilation_rate=1
    )

    x = TCNBlock(
        x,
        filters=32,
        dilation_rate=2
    )

    x = TCNBlock(
        x,
        filters=32,
        dilation_rate=4
    )

    # =========================================
    # LSTM
    # =========================================

    x = layers.LSTM(
        30,
        return_sequences=True
    )(x)

    x = layers.LSTM(
        30,
        return_sequences=False
    )(x)

    # =========================================
    # Fuzzy Neural Block
    # =========================================

    fuzzy = GaussianFuzzyBlock(
        units=16
    )(x)

    # =========================================
    # Classification
    # =========================================

    x = layers.Concatenate()([
        x,
        fuzzy
    ])

    x = layers.Dense(
        32,
        activation="elu"
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="Target_Probability"
    )(x)

    model = Model(
        inputs,
        outputs,
        name="EEG_TCFNet"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0001
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
model = build_EEG_TCFNet()

model.summary()

In [ ]:
available_subjects = [
    s for s in SELECTED_SUBJECTS
    if s in subject_data
]

print("Available subjects:", available_subjects)

if len(available_subjects) < 3:
    raise ValueError(
        "At least 3 subjects are required."
    )

In [ ]:
loso_results = []

all_true = []
all_pred = []
all_prob = []

history_list = []

for test_subject in available_subjects:

    print("\n")
    print("="*70)
    print("TEST SUBJECT:", test_subject)
    print("="*70)

    train_subjects = [
        s for s in available_subjects
        if s != test_subject
    ]

    # -----------------------------
    # Training data
    # -----------------------------

    X_train = np.concatenate(
        [
            subject_data[s]["X"]
            for s in train_subjects
        ],
        axis=0
    )

    y_train = np.concatenate(
        [
            subject_data[s]["y"]
            for s in train_subjects
        ],
        axis=0
    )

    # -----------------------------
    # Test data
    # -----------------------------

    X_test = subject_data[
        test_subject
    ]["X"]

    y_test = subject_data[
        test_subject
    ]["y"]

    print("Training:", X_train.shape)
    print("Testing :", X_test.shape)

    # -----------------------------
    # Normalize
    # -----------------------------

    scaler = StandardScaler()

    X_train_flat = X_train.reshape(
        X_train.shape[0],
        -1
    )

    X_test_flat = X_test.reshape(
        X_test.shape[0],
        -1
    )

    X_train_scaled = scaler.fit_transform(
        X_train_flat
    )

    X_test_scaled = scaler.transform(
        X_test_flat
    )

    X_train_scaled = X_train_scaled.reshape(
        X_train.shape
    )

    X_test_scaled = X_test_scaled.reshape(
        X_test.shape
    )

    # -----------------------------
    # Model input
    # -----------------------------

    X_train_model = prepare_for_model(
        X_train_scaled
    )

    X_test_model = prepare_for_model(
        X_test_scaled
    )

    # -----------------------------
    # Build fresh model
    # -----------------------------

    fold_model = build_EEG_TCFNet()

    # -----------------------------
    # Callbacks
    # -----------------------------

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7
    )

    # -----------------------------
    # Train
    # -----------------------------

    history = fold_model.fit(
        X_train_model,
        y_train,
        validation_split=0.2,
        epochs=15,
        batch_size=64,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=1
    )

    history_list.append(history)

    # -----------------------------
    # Prediction
    # -----------------------------

    probabilities = (
        fold_model.predict(
            X_test_model,
            verbose=0
        ).ravel()
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # -----------------------------
    # Metrics
    # -----------------------------

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    if len(np.unique(y_test)) == 2:
        auc = roc_auc_score(
            y_test,
            probabilities
        )
    else:
        auc = np.nan

    loso_results.append({
        "Test Subject": test_subject,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": auc
    })

    all_true.extend(y_test)
    all_pred.extend(predictions)
    all_prob.extend(probabilities)

    print("\nFold results:")
    print("Accuracy :", round(accuracy * 100, 2), "%")
    print("Precision:", round(precision * 100, 2), "%")
    print("Recall   :", round(recall * 100, 2), "%")
    print("F1       :", round(f1 * 100, 2), "%")
    print("ROC-AUC  :", round(auc, 4))

In [ ]:
results_df = pd.DataFrame(
    loso_results
)

display(results_df)

In [ ]:
all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_prob = np.array(all_prob)

overall_accuracy = accuracy_score(
    all_true,
    all_pred
)

overall_precision = precision_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_recall = recall_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_f1 = f1_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_auc = roc_auc_score(
    all_true,
    all_prob
)

print("="*60)
print("FINAL P300 CLASSIFICATION RESULTS")
print("="*60)

print(
    f"Accuracy  : {overall_accuracy*100:.2f}%"
)

print(
    f"Precision : {overall_precision*100:.2f}%"
)

print(
    f"Recall    : {overall_recall*100:.2f}%"
)

print(
    f"F1 Score  : {overall_f1*100:.2f}%"
)

print(
    f"ROC-AUC   : {overall_auc:.4f}"
)

In [ ]:
print(
    classification_report(
        all_true,
        all_pred,
        target_names=[
            "NonTarget",
            "Target"
        ],
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    all_true,
    all_pred
)

plt.figure(
    figsize=(6,5)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "NonTarget",
        "Target"
    ],
    yticklabels=[
        "NonTarget",
        "Target"
    ]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(
    "P300 EEG Classification Confusion Matrix"
)

plt.show()

In [ ]:
# Use first available subject for visualization

wave_subject = available_subjects[0]

X_wave = subject_data[
    wave_subject
]["X"]

y_wave = subject_data[
    wave_subject
]["y"]

target_epochs = X_wave[
    y_wave == 1
]

nontarget_epochs = X_wave[
    y_wave == 0
]

target_mean = target_epochs.mean(
    axis=0
).mean(axis=0)

nontarget_mean = nontarget_epochs.mean(
    axis=0
).mean(axis=0)

time = np.arange(
    EPOCH_SAMPLES
) / TARGET_SFREQ

plt.figure(
    figsize=(10,5)
)

plt.plot(
    time,
    target_mean,
    label="Target"
)

plt.plot(
    time,
    nontarget_mean,
    label="NonTarget"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EEG amplitude")
plt.title(
    "Target vs NonTarget EEG ERP Waveform"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(
    figsize=(9,5)
)

for i, history in enumerate(history_list):

    plt.plot(
        history.history["loss"],
        label=f"Fold {i+1} Training"
    )

    plt.plot(
        history.history["val_loss"],
        linestyle="--",
        label=f"Fold {i+1} Validation"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "EEG-TCFNet Training and Validation Loss"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
X_final = np.concatenate(
    [
        subject_data[s]["X"]
        for s in available_subjects
    ],
    axis=0
)

y_final = np.concatenate(
    [
        subject_data[s]["y"]
        for s in available_subjects
    ],
    axis=0
)

final_scaler = StandardScaler()

X_final_flat = X_final.reshape(
    X_final.shape[0],
    -1
)

X_final_scaled = final_scaler.fit_transform(
    X_final_flat
)

X_final_scaled = X_final_scaled.reshape(
    X_final.shape
)

X_final_model = prepare_for_model(
    X_final_scaled
)

final_model = build_EEG_TCFNet()

final_history = final_model.fit(
    X_final_model,
    y_final,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ],
    verbose=1
)

print("Final model training completed.")

In [ ]:
MODEL_PATH = "/content/EEG_TCFNet_final.keras"
SCALER_PATH = "/content/EEG_TCFNet_scaler.pkl"

final_model.save(
    MODEL_PATH
)

with open(
    SCALER_PATH,
    "wb"
) as f:
    pickle.dump(
        final_scaler,
        f
    )

print("Model saved:", MODEL_PATH)
print("Scaler saved:", SCALER_PATH)

In [ ]:
final_results = {
    "Subjects": available_subjects,
    "Model": "EEG-TCFNet",
    "Accuracy": float(overall_accuracy),
    "Precision": float(overall_precision),
    "Recall": float(overall_recall),
    "F1_Score": float(overall_f1),
    "ROC_AUC": float(overall_auc)
}

with open(
    "/content/final_results.json",
    "w"
) as f:

    json.dump(
        final_results,
        f,
        indent=4
    )

print("final_results.json saved.")

In [ ]:
APPLICATIONS = [
    "Light",
    "Fan",
    "AC",
    "TV"
]

# Select ONE application for this demonstration
SELECTED_APPLICATION = "Light"

print("Selected application:", SELECTED_APPLICATION)

In [ ]:
# Take one EEG sample from the dataset

demo_subject = available_subjects[0]

X_demo = subject_data[
    demo_subject
]["X"]

y_demo = subject_data[
    demo_subject
]["y"]

# Prefer a Target sample
target_indices = np.where(
    y_demo == 1
)[0]

if len(target_indices) == 0:
    raise ValueError(
        "No Target epoch available for smart application demo."
    )

demo_index = target_indices[0]

demo_epoch = X_demo[
    demo_index:demo_index+1
]

# Normalize
demo_flat = demo_epoch.reshape(
    1,
    -1
)

demo_scaled = final_scaler.transform(
    demo_flat
)

demo_scaled = demo_scaled.reshape(
    demo_epoch.shape
)

demo_input = prepare_for_model(
    demo_scaled
)

# Predict
probability = float(
    final_model.predict(
        demo_input,
        verbose=0
)[0][0]
)

prediction = int(
    probability >= 0.5
)

print("EEG Target probability:",
      round(probability, 4))

print(
    "Prediction:",
    "TARGET" if prediction == 1
    else "NON-TARGET"
)

In [ ]:
application_status = {}

for app in APPLICATIONS:

    application_status[app] = "OFF"

if prediction == 1:

    application_status[
        SELECTED_APPLICATION
    ] = "ON"

print("\n")
print("="*55)
print("       SMART APPLICATION CONTROL OUTPUT")
print("="*55)

for app, status in application_status.items():

    print(
        f"{app:<10} : {status}"
    )

print("="*55)

if prediction == 1:

    print(
        f"FINAL COMMAND : {SELECTED_APPLICATION} ON"
    )

else:

    print(
        "FINAL COMMAND : NO APPLICATION ACTIVATED"
    )

In [ ]:
SELECTED_APPLICATION = "Fan"

In [ ]:
smart_home_result = {
    "selected_application": SELECTED_APPLICATION,
    "prediction": (
        "Target"
        if prediction == 1
        else "NonTarget"
    ),
    "target_probability": probability,
    "applications": application_status
}

with open(
    "/content/smart_home_output.json",
    "w"
) as f:

    json.dump(
        smart_home_result,
        f,
        indent=4
    )

print("Smart-home result saved.")

In [ ]:
config = {
    "dataset_zip": "EEG1920.zip",
    "dataset_root": "/content/bigP3BCI",
    "subjects_used": available_subjects,
    "preprocessing": {
        "bandpass": "1-15 Hz",
        "notch": "50 Hz",
        "sampling_frequency": 120,
        "filter_order": 6,
        "winsorization": True
    },
    "classification": {
        "Target": 1,
        "NonTarget": 0
    },
    "model": {
        "name": "EEG-TCFNet",
        "CNN": True,
        "TCN": True,
        "LSTM": True,
        "LSTM_units": 30,
        "Fuzzy_Neural_Block": True,
        "optimizer": "Adam",
        "learning_rate": 0.0001
    },
    "evaluation": "Leave-One-Subject-Out",
    "smart_applications": APPLICATIONS
}

with open(
    "/content/project_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print("Project configuration saved.")

In [ ]:
print("\n")
print("="*70)
print("              FINAL PROJECT SUMMARY")
print("="*70)

print("Dataset          : BigP3BCI / NEMAR")
print("ZIP file         : EEG1920.zip")
print("Subjects         :", ", ".join(available_subjects))

print("\nPreprocessing")
print("------------------------------")
print("Bandpass         : 1-15 Hz")
print("Notch            : 50 Hz")
print("Sampling rate    : 120 Hz")
print("Butterworth      : 6th order")
print("Winsorization    : Yes")

print("\nModel")
print("------------------------------")
print("CNN              : Yes")
print("TCN              : Yes")
print("LSTM             : Yes")
print("FNB              : Yes")
print("Model            : EEG-TCFNet")
print("Optimizer        : Adam")
print("Learning rate    : 0.0001")

print("\nEvaluation")
print("------------------------------")
print("Method           : LOSO")
print(f"Accuracy         : {overall_accuracy*100:.2f}%")
print(f"Precision        : {overall_precision*100:.2f}%")
print(f"Recall           : {overall_recall*100:.2f}%")
print(f"F1 Score         : {overall_f1*100:.2f}%")
print(f"ROC-AUC          : {overall_auc:.4f}")

print("\nSmart Applications")
print("------------------------------")

for app, status in application_status.items():
    print(f"{app:<15}: {status}")

print("\nFINAL COMMAND")
print("------------------------------")

if prediction == 1:
    print(
        f"{SELECTED_APPLICATION} = ON"
    )
else:
    print(
        "No application activated"
    )

print("="*70)